# Phase 0: Setup and Installation
We mount Google Drive, clone the repository, and install required libraries.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Change working directory
%cd /content/drive/MyDrive/

# 3. Clone repo from GitHub
!git clone https://github.com/kienng221105/Voice_Interaction_For_Robot.git || true

# 4. Move into the project folder
%cd Voice_Interaction_For_Robot

# 5. Pull latest changes
!git pull

In [ ]:
!pip install -q transformers==4.46.3 datasets accelerate peft==0.13.2 trl==0.12.2 bitsandbytes scikit-learn psutil tqdm

# Phase 1: Dataset Preparation
Convert `data/raw/dataset_unified.jsonl` text data into structured Joint Intent + Slot Filling JSON format.

In [ ]:
import json
import re
import os

os.makedirs('data/processed', exist_ok=True)

aliases = {
    "coca": ["cô ca", "cô ka", "coca cô la", "co ca", "coca"],
    "pepsi": ["bép si", "pép xì", "pep xi", "pét si", "pepsi"],
    "7up": ["bảy áp", "se vừn ắp", "sê ven áp", "seven up", "7up"],
    "sting": ["xtinh", "xì tin", "x ting", "sting"],
    "aquafina": ["aqua fina", "a qua phi na", "ác qua", "aquafina"]
}

canonical_map = {}
for canon, variants in aliases.items():
    for variant in variants:
        canonical_map[variant] = canon

sorted_variants = sorted(canonical_map.keys(), key=len, reverse=True)

def parse_text_to_items(text, intent):
    items = []
    text_lower = text.lower()
    
    found_products = []
    for variant in sorted_variants:
        if variant in text_lower:
            canon = canonical_map[variant]
            if canon not in [p['product'] for p in found_products]:
                idx = text_lower.find(variant)
                prefix = text_lower[:idx]
                nums = re.findall(r'\b(\d+)\b', prefix)
                qty = int(nums[-1]) if nums else 1
                found_products.append({"product": canon, "quantity": qty, "idx": idx})
            text_lower = text_lower.replace(variant, " " * len(variant))
            
    found_products.sort(key=lambda x: x["idx"])
    
    for p in found_products:
        item = {"product": p["product"]}
        if intent not in ["cancel", "change_product"]: 
            item["quantity"] = p["quantity"]
        items.append(item)
        
    return items

train_data, val_data, test_data, unlabeled_data = [], [], [], []

with open('data/raw/dataset_unified.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        if not line.strip(): continue
        d = json.loads(line)
        split = d.get('split', 'unknown')
        text = d['text']
        
        if split == 'unlabeled':
            unlabeled_data.append({"text": text})
            continue
            
        intent = d['label']
        items = parse_text_to_items(text, intent)
        
        sample = {
            "text": text,
            "output": {"intent": intent, "items": items}
        }
        
        if split in ['train_gold', 'train_augmented']:
            train_data.append(sample)
        elif split == 'test_normal':
            val_data.append(sample)
        elif split == 'test_hard':
            test_data.append(sample)

def save_jsonl(data, path):
    with open(path, 'w', encoding='utf-8') as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

save_jsonl(train_data, 'data/processed/train.jsonl')
save_jsonl(val_data, 'data/processed/val.jsonl')
save_jsonl(test_data, 'data/processed/test.jsonl')
save_jsonl(unlabeled_data, 'data/processed/unlabeled.jsonl')

print(f"Prepared {len(train_data)} training samples.")
print(f"Prepared {len(val_data)} validation samples.")
print(f"Prepared {len(test_data)} test samples.")
print(f"Prepared {len(unlabeled_data)} unlabeled samples.")

# Phase 2 & 3: Teacher Labeling & Training Data
Use `Qwen2.5-7B-Instruct` to label unlabeled samples into structured JSON, then merge with gold data to form `train_distill.jsonl`.

In [ ]:
import torch
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from tqdm.notebook import tqdm

SYSTEM_PROMPT = """You are a Vietnamese Voice-Enabled Vending Machine Assistant.
Your task is Joint Intent Recognition and Entity Extraction (Slot Filling).
Extract the user's intent and product items into a structured JSON format.

Supported intents: greeting, show_menu, buy_product, add_product, change_product, payment, cancel, help, unknown
Canonical products: coca, pepsi, sting, aquafina, 7up (normalize all aliases)

Output JSON Format EXACTLY like this:
{
  "intent": "buy_product",
  "items": [
    {
      "product": "coca",
      "quantity": 2
    }
  ]
}

DO NOT output any explanation or markdown, just the raw JSON."""

def extract_json(text):
    text = text.strip()
    if "```json" in text: text = text.split("```json")[1].split("```")[0].strip()
    elif "```" in text: text = text.split("```")[1].split("```")[0].strip()
    try: return json.loads(text)
    except: return {"intent": "unknown", "items": []}

torch.cuda.empty_cache(); gc.collect()

print("Loading Teacher model...")
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
teacher_id = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(teacher_id)
model = AutoModelForCausalLM.from_pretrained(teacher_id, quantization_config=bnb_config, device_map="auto")

pseudo_labeled = []
for item in tqdm(unlabeled_data, desc="Teacher Labeling"):
    text = item['text']
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Text: \"{text}\""}
    ]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    outputs = model.generate(**inputs, max_new_tokens=150, do_sample=False)
    resp = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    
    pseudo_labeled.append({"text": text, "output": extract_json(resp)})

save_jsonl(pseudo_labeled, 'data/processed/pseudo_labeled.jsonl')

train_distill = train_data + pseudo_labeled
save_jsonl(train_distill, 'data/processed/train_distill.jsonl')

del model, tokenizer; torch.cuda.empty_cache(); gc.collect()
print(f"Teacher labeling done. Final training data size: {len(train_distill)}")

# Phase 4 & 5: Student Fine-Tuning (QLoRA)
Fine-tune `Qwen2.5-1.5B-Instruct` on `train_distill.jsonl`. We enforce strict JSON generation.

In [ ]:
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

STUDENT = 'Qwen/Qwen2.5-1.5B-Instruct'
s_tok = AutoTokenizer.from_pretrained(STUDENT)
s_tok.pad_token = s_tok.eos_token

rows = []
for item in train_distill:
    msgs = [
        {'role':'system',   'content': SYSTEM_PROMPT},
        {'role':'user',     'content': f'Text: "{item["text"]}"'},
        {'role':'assistant','content': json.dumps(item["output"], ensure_ascii=False)},
    ]
    rows.append({'text': s_tok.apply_chat_template(msgs, tokenize=False)})
train_dataset = Dataset.from_list(rows)

bnb4 = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)
s_model = AutoModelForCausalLM.from_pretrained(STUDENT, quantization_config=bnb4, device_map='auto', torch_dtype=torch.float16)
s_model = prepare_model_for_kbit_training(s_model)

lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj'],
    bias='none', task_type='CAUSAL_LM',
)
s_model = get_peft_model(s_model, lora_cfg)

# Fix T4 crash (prevent bfloat16 gradients)
for name, param in s_model.named_parameters():
    if param.data.dtype == torch.bfloat16: param.data = param.data.to(torch.float16)
for name, buf in s_model.named_buffers():
    if buf.dtype == torch.bfloat16: buf.data = buf.data.to(torch.float16)

training_args = SFTConfig(
    dataset_text_field='text', output_dir='./student_1.5b_qwen',
    per_device_train_batch_size=2, gradient_accumulation_steps=8,
    learning_rate=2e-4, num_train_epochs=3, logging_steps=10,
    save_strategy='epoch', optim='paged_adamw_32bit',
    fp16=False, bf16=False, report_to='none',
)

trainer = SFTTrainer(model=s_model, train_dataset=train_dataset, processing_class=s_tok, args=training_args)

print(f'Starting student fine-tuning on {len(train_dataset)} samples...')
trainer.train()

trainer.model.save_pretrained('./student_1.5b_adapter')
s_tok.save_pretrained('./student_1.5b_adapter')
print('Adapter saved.')

del trainer, s_model; torch.cuda.empty_cache(); gc.collect()

# Phase 6: Evaluation
Compute Accuracy, Macro F1 for Intent, and End-to-End correctness (Intent + all products + all quantities).

In [ ]:
from peft import PeftModel
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

def extract_entities(items):
    # Returns a list of strings representing entities: "product_quantity"
    entities = []
    for item in items:
        p = item.get('product', '')
        q = item.get('quantity', 'N/A') # Use N/A if missing
        entities.append(f"{p}_{q}")
    return entities

def match_items(pred_items, true_items):
    def sort_key(x): return x.get('product', '')
    p_items = sorted(pred_items, key=sort_key)
    t_items = sorted(true_items, key=sort_key)
    if len(p_items) != len(t_items): return False
    for p, t in zip(p_items, t_items):
        if p.get('product') != t.get('product'): return False
        if 'quantity' in t and p.get('quantity') != t.get('quantity'): return False
    return True

def evaluate(model, tokenizer, data, label):
    y_true_intent, y_pred_intent = [], []
    e2e_correct = 0
    errors = []
    
    # For Entity metrics
    total_true_entities = 0
    total_pred_entities = 0
    correct_entities = 0
    
    for item in tqdm(data, desc=label):
        text = item['text']
        true_out = item['output']
        
        msgs = [
            {'role':'system','content': SYSTEM_PROMPT},
            {'role':'user',  'content': f'Text: "{text}"'},
        ]
        inp = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        ids = tokenizer([inp], return_tensors='pt').to(model.device)
        out = model.generate(**ids, max_new_tokens=150, do_sample=False)
        resp = tokenizer.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True)
        
        pred_out = extract_json(resp)
        
        y_true_intent.append(true_out.get('intent', 'unknown'))
        y_pred_intent.append(pred_out.get('intent', 'unknown'))
        
        # End-to-End calculation
        intent_match = true_out.get('intent') == pred_out.get('intent')
        items_match = match_items(pred_out.get('items', []), true_out.get('items', []))
        
        if intent_match and items_match:
            e2e_correct += 1
        else:
            errors.append({"text": text, "true": true_out, "pred": pred_out})
            
        # Entity Precision / Recall Calculation
        t_ents = extract_entities(true_out.get('items', []))
        p_ents = extract_entities(pred_out.get('items', []))
        
        total_true_entities += len(t_ents)
        total_pred_entities += len(p_ents)
        
        # Count intersection
        for p_e in p_ents:
            if p_e in t_ents:
                correct_entities += 1
                t_ents.remove(p_e) # remove to handle duplicates properly
            
    # Intent Metrics
    acc = accuracy_score(y_true_intent, y_pred_intent)
    f1 = f1_score(y_true_intent, y_pred_intent, average='macro', zero_division=0)
    
    # Detailed classification report
    report = classification_report(y_true_intent, y_pred_intent, zero_division=0)
    
    # Entity Metrics
    ent_precision = correct_entities / total_pred_entities if total_pred_entities > 0 else 0
    ent_recall = correct_entities / total_true_entities if total_true_entities > 0 else 0
    ent_f1 = (2 * ent_precision * ent_recall) / (ent_precision + ent_recall) if (ent_precision + ent_recall) > 0 else 0
    
    e2e_acc = e2e_correct / len(data) if len(data) > 0 else 0
    
    print(f'\n{"="*40}')
    print(f'{label.upper()} RESULTS')
    print(f'{"="*40}\n')
    
    print('--- INTENT METRICS ---')
    print(f'Overall Accuracy : {acc:.4f}')
    print(f'Macro F1 Score   : {f1:.4f}\n')
    print('Classification Report:')
    print(report)
    
    print('--- ENTITY METRICS (Strict Match) ---')
    print(f'Entity Precision : {ent_precision:.4f}')
    print(f'Entity Recall    : {ent_recall:.4f}')
    print(f'Entity F1 Score  : {ent_f1:.4f}\n')
    
    print('--- SYSTEM METRICS ---')
    print(f'End-to-End Accuracy: {e2e_acc:.4f} (Strict Intent + Full Items Match)\n')
    
    return errors

base = AutoModelForCausalLM.from_pretrained(STUDENT, quantization_config=bnb4, device_map='auto', torch_dtype=torch.float16)
eval_model = PeftModel.from_pretrained(base, './student_1.5b_adapter')
eval_tok   = AutoTokenizer.from_pretrained('./student_1.5b_adapter')

normal_errors = evaluate(eval_model, eval_tok, val_data, 'Normal Test')
hard_errors = evaluate(eval_model, eval_tok, test_data, 'Hard Test')

# Phase 7: Error Analysis
print("\n=== PHASE 7: HARD TEST ERROR ANALYSIS ===")
for err in hard_errors[:10]:
    print(f"\nTEXT: {err['text']}")
    print(f"TRUE: {json.dumps(err['true'], ensure_ascii=False)}")
    print(f"PRED: {json.dumps(err['pred'], ensure_ascii=False)}")
    
del eval_model, base; torch.cuda.empty_cache(); gc.collect()


# Phase 8: CPU Benchmark
Test latency and memory footprint of the Student model on CPU.

In [ ]:
import time, psutil
cpu_base = AutoModelForCausalLM.from_pretrained(STUDENT, device_map='cpu', torch_dtype=torch.float32)
cpu_model = PeftModel.from_pretrained(cpu_base, './student_1.5b_adapter')

mem_mb = psutil.Process().memory_info().rss / 1024**2
print(f'\nCPU Model loaded. RAM usage: {mem_mb:.0f} MB')

def latency(text):
    msgs = [
        {'role':'system','content': SYSTEM_PROMPT},
        {'role':'user',  'content': f'Text: "{text}"'},
    ]
    inp = eval_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = eval_tok([inp], return_tensors='pt')
    t0 = time.time()
    _ = cpu_model.generate(**ids, max_new_tokens=150, do_sample=False)
    return time.time() - t0

tests = [
    'cho 2 coca và 1 pepsi',
    'đổi sang aqua fina rồi trả tiền',
]
# Warmup
latency(tests[0])

lats = [latency(t) for t in tests]
print(f'\nAvg CPU latency: {sum(lats)/len(lats):.3f} s/query')
for t, l in zip(tests, lats):
    print(f'  {l:.3f}s | {t}')

# Phase 9: Export to GGUF (Tối ưu hóa cực hạn cho CPU)
Nếu chạy Inference bằng PyTorch trên CPU mất 20s, việc chuyển model sang định dạng **GGUF (Q4_K_M)** sẽ giảm thời gian xuống còn **< 1s**.
Quy trình gồm 2 bước:
1. **Merge Model:** Nấu chảy Adapter (LoRA) dính chặt vào Base Model.
2. **Quantize GGUF:** Dùng công cụ `llama.cpp` để ép model xuống 4-bit chuẩn GGUF.

In [ ]:
import torch
import gc
import os
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# Kiểm tra xem file fp16 (bản gốc 3.1GB) đã được tạo từ trước chưa
if not os.path.exists("./student_1.5b_fp16.gguf"):
    print("File FP16 chưa tồn tại. Bắt đầu Merge Model (Sẽ tốn nhiều RAM)...")
    base_model = AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen2.5-1.5B-Instruct", 
        device_map="cpu", 
        torch_dtype=torch.float16
    )
    model_peft = PeftModel.from_pretrained(base_model, "./student_1.5b_adapter")
    merged_model = model_peft.merge_and_unload()

    merged_model.save_pretrained("./student_1.5b_merged")
    tokenizer = AutoTokenizer.from_pretrained("./student_1.5b_adapter")
    tokenizer.save_pretrained("./student_1.5b_merged")

    # Giải phóng RAM ngay lập tức
    del merged_model, model_peft, base_model, tokenizer
    gc.collect()
    
    print("Tải llama.cpp và Convert sang FP16 GGUF...")
    !git clone https://github.com/ggerganov/llama.cpp.git || true
    %cd llama.cpp
    !pip install -q gguf
    !python convert_hf_to_gguf.py ../student_1.5b_merged --outfile ../student_1.5b_fp16.gguf --outtype f16
    %cd ..
else:
    print("✅ Đã tìm thấy file student_1.5b_fp16.gguf trên Drive!")
    print("Bỏ qua toàn bộ bước Merge RAM nặng nề, tiến thẳng tới Quantize 8-bit.")
    if not os.path.exists("llama.cpp"):
        !git clone https://github.com/ggerganov/llama.cpp.git || true

print("\n--- BƯỚC 2: BUILD LLAMA.CPP VÀ QUANTIZE ---")
%cd llama.cpp
print("Biên dịch C++ (CMake)...")
!cmake -B build
!cmake --build build --config Release -j

print("\nĐang ép model xuống 8-bit (Q8_0)...")
!./build/bin/llama-quantize ../student_1.5b_fp16.gguf ../student_1.5b_q8_0.gguf q8_0
%cd ..

print("\n🎉 HOÀN TẤT! File siêu chuẩn xác của bạn là: student_1.5b_q8_0.gguf")



# Phase 10: Benchmark GGUF 4-bit trên CPU của Colab
Chúng ta sẽ cài đặt `llama-cpp-python` và load file `.gguf` vừa tạo để xem tốc độ trên CPU thực tế ra sao (Colab cung cấp sẵn 2 cores CPU).

In [ ]:
# Cài đặt thư viện chạy GGUF trên Python
!pip install -q llama-cpp-python

import time
import json
from llama_cpp import Llama

print("Đang load model GGUF 8-bit vào RAM...")
# Khởi tạo model
llm = Llama(
    model_path="./student_1.5b_q8_0.gguf",
    n_gpu_layers=0, # Ép chạy 100% bằng CPU
    n_ctx=512,      # Context window nhỏ cho SLU
    verbose=False   # Tắt bớt log rác
)
print("Load thành công!\n")

SYSTEM_PROMPT = """You are a Vietnamese Voice-Enabled Vending Machine Assistant.
Your task is Joint Intent Recognition and Entity Extraction (Slot Filling).
Extract the user's intent and product items into a structured JSON format.

Supported intents: greeting, show_menu, buy_product, add_product, change_product, payment, cancel, help, unknown
Canonical products: coca, pepsi, sting, aquafina, 7up (normalize all aliases)

Output JSON Format EXACTLY like this:
{
  "intent": "buy_product",
  "items": [
    {
      "product": "coca",
      "quantity": 2
    }
  ]
}

DO NOT output any explanation or markdown, just the raw JSON."""

def predict_gguf(text):
    # Format prompt chuẩn của Qwen Chat
    prompt = f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\nText: \"{text}\"<|im_end|>\n<|im_start|>assistant\n"
    
    t0 = time.time()
    response = llm(
        prompt,
        max_tokens=150,
        stop=["<|im_end|>"],
        echo=False,
        temperature=0.0
    )
    latency = time.time() - t0
    
    raw_text = response['choices'][0]['text'].strip()
    
    if "```json" in raw_text: raw_text = raw_text.split("```json")[1].split("```")[0].strip()
    elif "```" in raw_text: raw_text = raw_text.split("```")[1].split("```")[0].strip()
        
    try:
        parsed = json.loads(raw_text)
    except:
        parsed = {"error": "Failed JSON parse", "raw": raw_text}
        
    return parsed, latency

tests = [
    "cho 2 chai nước suối a qua phi na",
    "thôi đổi ý không lấy nước suối nữa, lấy 1 lon bép si",
    "tính tiền luôn đi"
]

print("=== BẮT ĐẦU TEST TỐC ĐỘ CPU (GGUF 8-Bit) ===\n")
# Chạy warmup 1 câu để làm nóng cache CPU
_, _ = predict_gguf("warmup")

for t in tests:
    print(f"🗣️ User: {t}")
    res, lat = predict_gguf(t)
    print(f"⏱️ Latency: {lat:.3f} giây")
    print(f"🤖 Assistant: {json.dumps(res, ensure_ascii=False)}\n")
    print("-"*40)


# Phase 11: Đánh giá Toàn diện Model 4-bit (Q4_K_M) trên tập Test

In [ ]:
import json
import time
from tqdm import tqdm
from llama_cpp import Llama
from sklearn.metrics import accuracy_score, f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# Khởi tạo mô hình
print("Đang load model ./student_1.5b_q4_k_m.gguf vào RAM CPU...")
llm_gguf = Llama(
    model_path="./student_1.5b_q4_k_m.gguf",
    n_gpu_layers=0,
    n_ctx=512,
    verbose=False
)
print("Load thành công! Bắt đầu đánh giá toàn diện...")

def extract_entities(items):
    entities = []
    for item in items:
        p = item.get('product', '')
        q = item.get('quantity', 'N/A')
        entities.append(f"{p}_{q}")
    return entities

def match_items(pred_items, true_items):
    def sort_key(x): return x.get('product', '')
    p_items = sorted(pred_items, key=sort_key)
    t_items = sorted(true_items, key=sort_key)
    if len(p_items) != len(t_items): return False
    for p, t in zip(p_items, t_items):
        if p.get('product') != t.get('product'): return False
        if 'quantity' in t and p.get('quantity') != t.get('quantity'): return False
    return True

def evaluate_gguf(data, label):
    y_true_intent, y_pred_intent = [], []
    e2e_correct = 0
    total_true_entities, total_pred_entities, correct_entities = 0, 0, 0
    total_latency = 0
    
    for item in tqdm(data, desc=label):
        text = item['text']
        true_out = item['output']
        
        prompt = f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\nText: \"{text}\"<|im_end|>\n<|im_start|>assistant\n"
        
        t0 = time.time()
        response = llm_gguf(prompt, max_tokens=150, stop=["<|im_end|>"], echo=False, temperature=0.0)
        total_latency += time.time() - t0
        
        raw_text = response['choices'][0]['text'].strip()
        if "```json" in raw_text: raw_text = raw_text.split("```json")[1].split("```")[0].strip()
        elif "```" in raw_text: raw_text = raw_text.split("```")[1].split("```")[0].strip()
            
        try: pred_out = json.loads(raw_text)
        except: pred_out = {}
            
        y_true_intent.append(true_out.get('intent', 'unknown'))
        y_pred_intent.append(pred_out.get('intent', 'unknown'))
        
        intent_match = true_out.get('intent') == pred_out.get('intent')
        items_match = match_items(pred_out.get('items', []), true_out.get('items', []))
        if intent_match and items_match: e2e_correct += 1
            
        t_ents = extract_entities(true_out.get('items', []))
        p_ents = extract_entities(pred_out.get('items', []))
        
        total_true_entities += len(t_ents)
        total_pred_entities += len(p_ents)
        
        for p_e in p_ents:
            if p_e in t_ents:
                correct_entities += 1
                t_ents.remove(p_e)
                
    acc = accuracy_score(y_true_intent, y_pred_intent)
    f1 = f1_score(y_true_intent, y_pred_intent, average='macro', zero_division=0)
    report = classification_report(y_true_intent, y_pred_intent, zero_division=0)
    
    ent_precision = correct_entities / total_pred_entities if total_pred_entities > 0 else 0
    ent_recall = correct_entities / total_true_entities if total_true_entities > 0 else 0
    ent_f1 = (2 * ent_precision * ent_recall) / (ent_precision + ent_recall) if (ent_precision + ent_recall) > 0 else 0
    
    e2e_acc = e2e_correct / len(data) if len(data) > 0 else 0
    avg_latency = total_latency / len(data) if len(data) > 0 else 0
    
    print(f'\n{"="*40}')
    print(f'{label.upper()} RESULTS - ./student_1.5b_q4_k_m.gguf')
    print(f'{"="*40}\n')
    
    print('--- INTENT METRICS ---')
    print(f'Overall Accuracy : {acc:.4f}')
    print(f'Macro F1 Score   : {f1:.4f}\n')
    print('Classification Report:')
    print(report)
    
    print('--- ENTITY METRICS (Strict Match) ---')
    print(f'Entity Precision : {ent_precision:.4f}')
    print(f'Entity Recall    : {ent_recall:.4f}')
    print(f'Entity F1 Score  : {ent_f1:.4f}\n')
    
    print('--- SYSTEM METRICS ---')
    print(f'End-to-End Accuracy: {e2e_acc:.4f} (Strict Intent + Full Items Match)')
    print(f'Average CPU Latency: {avg_latency:.3f} seconds/query\n')

# Chạy test trên tập Validation và Hard Test
evaluate_gguf(val_data, 'Normal Test (GGUF)')
evaluate_gguf(test_data, 'Hard Test (GGUF)')

# Giải phóng RAM cho model tiếp theo
del llm_gguf


# Phase 12: Đánh giá Toàn diện Model 8-bit (Q8_0) trên tập Test
Kiểm tra xem bản 8-bit có chống được lỗi ảo giác (hallucination) từ vụ 'bép si -> sting' hay không.

In [ ]:
import json
import time
from tqdm import tqdm
from llama_cpp import Llama
from sklearn.metrics import accuracy_score, f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# Khởi tạo mô hình
print("Đang load model ./student_1.5b_q8_0.gguf vào RAM CPU...")
llm_gguf = Llama(
    model_path="./student_1.5b_q8_0.gguf",
    n_gpu_layers=0,
    n_ctx=512,
    verbose=False
)
print("Load thành công! Bắt đầu đánh giá toàn diện...")

def extract_entities(items):
    entities = []
    for item in items:
        p = item.get('product', '')
        q = item.get('quantity', 'N/A')
        entities.append(f"{p}_{q}")
    return entities

def match_items(pred_items, true_items):
    def sort_key(x): return x.get('product', '')
    p_items = sorted(pred_items, key=sort_key)
    t_items = sorted(true_items, key=sort_key)
    if len(p_items) != len(t_items): return False
    for p, t in zip(p_items, t_items):
        if p.get('product') != t.get('product'): return False
        if 'quantity' in t and p.get('quantity') != t.get('quantity'): return False
    return True

def evaluate_gguf(data, label):
    y_true_intent, y_pred_intent = [], []
    e2e_correct = 0
    total_true_entities, total_pred_entities, correct_entities = 0, 0, 0
    total_latency = 0
    
    for item in tqdm(data, desc=label):
        text = item['text']
        true_out = item['output']
        
        prompt = f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\nText: \"{text}\"<|im_end|>\n<|im_start|>assistant\n"
        
        t0 = time.time()
        response = llm_gguf(prompt, max_tokens=150, stop=["<|im_end|>"], echo=False, temperature=0.0)
        total_latency += time.time() - t0
        
        raw_text = response['choices'][0]['text'].strip()
        if "```json" in raw_text: raw_text = raw_text.split("```json")[1].split("```")[0].strip()
        elif "```" in raw_text: raw_text = raw_text.split("```")[1].split("```")[0].strip()
            
        try: pred_out = json.loads(raw_text)
        except: pred_out = {}
            
        y_true_intent.append(true_out.get('intent', 'unknown'))
        y_pred_intent.append(pred_out.get('intent', 'unknown'))
        
        intent_match = true_out.get('intent') == pred_out.get('intent')
        items_match = match_items(pred_out.get('items', []), true_out.get('items', []))
        if intent_match and items_match: e2e_correct += 1
            
        t_ents = extract_entities(true_out.get('items', []))
        p_ents = extract_entities(pred_out.get('items', []))
        
        total_true_entities += len(t_ents)
        total_pred_entities += len(p_ents)
        
        for p_e in p_ents:
            if p_e in t_ents:
                correct_entities += 1
                t_ents.remove(p_e)
                
    acc = accuracy_score(y_true_intent, y_pred_intent)
    f1 = f1_score(y_true_intent, y_pred_intent, average='macro', zero_division=0)
    report = classification_report(y_true_intent, y_pred_intent, zero_division=0)
    
    ent_precision = correct_entities / total_pred_entities if total_pred_entities > 0 else 0
    ent_recall = correct_entities / total_true_entities if total_true_entities > 0 else 0
    ent_f1 = (2 * ent_precision * ent_recall) / (ent_precision + ent_recall) if (ent_precision + ent_recall) > 0 else 0
    
    e2e_acc = e2e_correct / len(data) if len(data) > 0 else 0
    avg_latency = total_latency / len(data) if len(data) > 0 else 0
    
    print(f'\n{"="*40}')
    print(f'{label.upper()} RESULTS - ./student_1.5b_q8_0.gguf')
    print(f'{"="*40}\n')
    
    print('--- INTENT METRICS ---')
    print(f'Overall Accuracy : {acc:.4f}')
    print(f'Macro F1 Score   : {f1:.4f}\n')
    print('Classification Report:')
    print(report)
    
    print('--- ENTITY METRICS (Strict Match) ---')
    print(f'Entity Precision : {ent_precision:.4f}')
    print(f'Entity Recall    : {ent_recall:.4f}')
    print(f'Entity F1 Score  : {ent_f1:.4f}\n')
    
    print('--- SYSTEM METRICS ---')
    print(f'End-to-End Accuracy: {e2e_acc:.4f} (Strict Intent + Full Items Match)')
    print(f'Average CPU Latency: {avg_latency:.3f} seconds/query\n')

# Chạy test trên tập Validation và Hard Test
evaluate_gguf(val_data, 'Normal Test (GGUF)')
evaluate_gguf(test_data, 'Hard Test (GGUF)')

# Giải phóng RAM cho model tiếp theo
del llm_gguf


# Phase 13: Tải Model 8-bit về máy tính (hoặc Kiosk)
Chạy ô này để tải trực tiếp file `student_1.5b_q8_0.gguf` về trình duyệt của bạn.

In [ ]:
from google.colab import files
import os

file_path = './student_1.5b_q8_0.gguf'
if os.path.exists(file_path):
    print("Đang nén và chuẩn bị tải file 1.6GB về máy...")
    files.download(file_path)
else:
    print("Lỗi: Không tìm thấy file 8-bit! Hãy đảm bảo bạn đã chạy Phase 9 thành công.")


# Phase 14: Đánh giá Model Gốc (HuggingFace Merged Model)
Đây là phiên bản PyTorch gốc 100% của Student Model sau khi Distill (Chưa qua bất kỳ bước nén GGUF nào).


In [ ]:
import torch
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm
import time
import json

print("Đang load HuggingFace Merged Model vào RAM...")
tokenizer_orig = AutoTokenizer.from_pretrained("./student_1.5b_merged")
model_orig = AutoModelForCausalLM.from_pretrained(
    "./student_1.5b_merged",
    device_map="auto",
    torch_dtype=torch.float16
)
model_orig.eval()

def evaluate_hf(data, label):
    y_true_intent, y_pred_intent = [], []
    e2e_correct = 0
    total_true_entities, total_pred_entities, correct_entities = 0, 0, 0
    total_latency = 0
    
    for item in tqdm(data, desc=label):
        text = item['text']
        true_out = item['output']
        
        prompt = f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\nText: \"{text}\"<|im_end|>\n<|im_start|>assistant\n"
        inputs = tokenizer_orig(prompt, return_tensors="pt").to(model_orig.device)
        
        t0 = time.time()
        with torch.no_grad():
            outputs = model_orig.generate(**inputs, max_new_tokens=150, temperature=0.0)
        total_latency += time.time() - t0
        
        generated_text = tokenizer_orig.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        raw_text = generated_text.strip()
        
        if "```json" in raw_text: raw_text = raw_text.split("```json")[1].split("```")[0].strip()
        elif "```" in raw_text: raw_text = raw_text.split("```")[1].split("```")[0].strip()
            
        try: pred_out = json.loads(raw_text)
        except: pred_out = {}
            
        y_true_intent.append(true_out.get('intent', 'unknown'))
        y_pred_intent.append(pred_out.get('intent', 'unknown'))
        
        intent_match = true_out.get('intent') == pred_out.get('intent')
        items_match = match_items(pred_out.get('items', []), true_out.get('items', []))
        if intent_match and items_match: e2e_correct += 1
            
        t_ents = extract_entities(true_out.get('items', []))
        p_ents = extract_entities(pred_out.get('items', []))
        
        total_true_entities += len(t_ents)
        total_pred_entities += len(p_ents)
        for p_e in p_ents:
            if p_e in t_ents:
                correct_entities += 1
                t_ents.remove(p_e)
                
    from sklearn.metrics import accuracy_score, f1_score, classification_report
    acc = accuracy_score(y_true_intent, y_pred_intent)
    f1 = f1_score(y_true_intent, y_pred_intent, average='macro', zero_division=0)
    
    ent_precision = correct_entities / total_pred_entities if total_pred_entities > 0 else 0
    ent_recall = correct_entities / total_true_entities if total_true_entities > 0 else 0
    ent_f1 = (2 * ent_precision * ent_recall) / (ent_precision + ent_recall) if (ent_precision + ent_recall) > 0 else 0
    e2e_acc = e2e_correct / len(data) if len(data) > 0 else 0
    avg_latency = total_latency / len(data) if len(data) > 0 else 0
    
    print(f'\n{"="*40}\n{label.upper()} RESULTS - HUGGINGFACE MERGED\n{"="*40}\n')
    print(f'Macro F1 Score     : {f1:.4f}')
    print(f'Entity F1 Score    : {ent_f1:.4f}')
    print(f'End-to-End Accuracy: {e2e_acc:.4f}')
    print(f'Average GPU Latency: {avg_latency:.3f} seconds/query\n')

evaluate_hf(val_data, 'Normal Test (HF Merged)')
evaluate_hf(test_data, 'Hard Test (HF Merged)')

del model_orig, tokenizer_orig
gc.collect()
torch.cuda.empty_cache()


# Phase 15: Đánh giá Model GGUF FP16 (Bản GGUF Không Nén)
File `student_1.5b_fp16.gguf` là bản dịch trực tiếp từ Merged Model sang định dạng của llama.cpp, không hề bị nén (chỉ thay đổi định dạng file). Độ chính xác sẽ y hệt như Phase 14, nhưng chạy được trên CPU.


In [ ]:
import gc
from llama_cpp import Llama

evaluate_gguf_code_template = ... # Dùng lại hàm evaluate_gguf từ Phase 11/12

print("Đang load model FP16 GGUF (3.1GB) vào RAM CPU...")
llm_gguf = Llama(
    model_path="./student_1.5b_fp16.gguf",
    n_gpu_layers=0,
    n_ctx=512,
    verbose=False
)

evaluate_gguf(val_data, 'Normal Test (GGUF FP16)')
evaluate_gguf(test_data, 'Hard Test (GGUF FP16)')

del llm_gguf
gc.collect()
